# Pipeline de execução

Coorte: `csvs/adnimerged_longitudinal.csv` (407 pacientes × 3 visitas: baseline, m12, m24).

1. `resample.py`
2. `feat_gen_dvf.py` — warps ANTs (template CN → imagem clínica)
3. `feat_vol.py` → `csvs/longitudinal_4_groups/features_volumetric.csv`
4. `feat_rad.py` → `csvs/longitudinal_4_groups/features_radiomic.csv`
5. `feat_dvf.py` → `csvs/longitudinal_4_groups/features_displacement.csv` (legado + momentos do artigo)
6. Este notebook — merge, wide T1/T2/T3, filtros e deltas

In [1]:
import pandas as pd

COHORT_PATH = "csvs/adnimerged_longitudinal.csv"
SLOT_ORDER = {"baseline": 0, "m12": 1, "m24": 2}

df_ext = pd.read_csv(COHORT_PATH)
if "slot" in df_ext.columns:
    df_ext["_slot_ord"] = df_ext["slot"].map(SLOT_ORDER).fillna(99)
    sort_cols = ["ID_PT", "_slot_ord", "MRI_DATE", "ID_IMG"]
else:
    sort_cols = ["ID_PT", "MRI_DATE", "ID_IMG"]

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(sort_cols)
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 407
Linhas totais: 1221
Checagem linhas/3: 407.0

Por GROUP:
GROUP
AD       83
CN      136
pMCI     81
sMCI    107
Name: count, dtype: int64

Por SEX:
SEX
F    184
M    223
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
AD      41   42   83
CN      72   64  136
pMCI    33   48   81
sMCI    38   69  107
All    184  223  407


# PLanilha dos atributos

In [ ]:
# Leitura radiomicos e volumetricos

import pandas as pd

ab = "longitudinal_4_groups"
BASE = f"csvs/{ab}"

vol_path = f"{BASE}/features_volumetric.csv"
rad_path = f"{BASE}/features_radiomic.csv"

vol_df = pd.read_csv(vol_path)
rad_df = pd.read_csv(rad_path)

print(vol_df.columns.tolist())
print(rad_df.columns.tolist())


['ID_IMG', 'roi', 'side', 'label', 'mask_mm3', 'gm_mm3', 'gm_norm', 'wm_mm3', 'wm_norm', 'csf_mm3', 'csf_norm', 'tissues_mm3', 'tissues_norm']
['ID_IMG', 'roi', 'side', 'label', 'original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Median', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_Autocorrelation', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_glcm_Correlation', 'original_gl

In [35]:
# Une radiomicos + volumetricos (por ROI) e normaliza pelo ICV o que depende de tamanho

MERGE_KEYS = ["ID_IMG", "roi", "side", "label"]

VOL_FEAT_COLS = [
    "mask_mm3",
    "gm_mm3",
    "gm_norm",
    "wm_mm3",
    "wm_norm",
    "csf_mm3",
    "csf_norm",
    "tissues_mm3",
    "tissues_norm",
]

RAD_SIZE_COLS = [
    "original_firstorder_Energy",
    "original_firstorder_TotalEnergy",
    "original_shape_MeshVolume",
    "original_shape_VoxelVolume",
    "original_shape_SurfaceArea",
    "original_shape_LeastAxisLength",
    "original_shape_MajorAxisLength",
    "original_shape_MinorAxisLength",
    "original_shape_Maximum2DDiameterColumn",
    "original_shape_Maximum2DDiameterRow",
    "original_shape_Maximum2DDiameterSlice",
    "original_shape_Maximum3DDiameter",
]

VOL_SIZE_COLS = ["mask_mm3", "gm_mm3", "wm_mm3", "csf_mm3", "tissues_mm3"]


def normalize_merge_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ID_IMG"] = out["ID_IMG"].astype(str).str.strip()
    out["roi"] = out["roi"].astype(str).str.strip()
    out["side"] = out["side"].astype(str).str.strip()
    label_num = pd.to_numeric(out["label"], errors="coerce")
    out["label"] = label_num.map(lambda x: str(int(x)) if pd.notna(x) else "")
    return out


# 1) ICV por ID_IMG (linha __global__)
vol_df = normalize_merge_keys(vol_df)

icv_df = (
    vol_df.loc[vol_df["roi"] == "__global__", ["ID_IMG", "mask_mm3"]]
    .drop_duplicates(subset=["ID_IMG"], keep="last")
    .rename(columns={"mask_mm3": "ICV_mask_mm3"})
)
icv_df["ICV_mask_mm3"] = pd.to_numeric(icv_df["ICV_mask_mm3"], errors="coerce")

# 2) features volumetricas por ROI (sem __global__)
vol_roi = (
    vol_df.loc[vol_df["roi"] != "__global__", MERGE_KEYS + VOL_FEAT_COLS]
    .drop_duplicates(subset=MERGE_KEYS, keep="last")
)
for c in VOL_FEAT_COLS:
    vol_roi[c] = pd.to_numeric(vol_roi[c], errors="coerce")

# 3) merge radiomicos + volumetricos + ICV
rad_df = normalize_merge_keys(rad_df)

merged = rad_df.merge(vol_roi, on=MERGE_KEYS, how="left", validate="one_to_one")
merged = merged.merge(icv_df, on="ID_IMG", how="left", validate="many_to_one")

missing_vol = int(merged[VOL_FEAT_COLS].isna().all(axis=1).sum())
if missing_vol:
    raise ValueError(
        f"Volumetria ausente para {missing_vol} linhas radiomicas. "
        "Verifique se (ID_IMG, roi, side, label) bate entre os dois CSVs."
    )

missing_icv = int(merged["ICV_mask_mm3"].isna().sum())
if missing_icv:
    raise ValueError(
        f"ICV ausente para {missing_icv} linhas radiomicas. "
        "Verifique se todos os ID_IMG do radiomico existem no volumetrico (linha __global__)."
    )

# 4) normaliza pelo ICV apenas medidas absolutas (mm3)
cols_to_norm = [c for c in RAD_SIZE_COLS + VOL_SIZE_COLS if c in merged.columns]
for c in cols_to_norm:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

merged[cols_to_norm] = merged[cols_to_norm].div(merged["ICV_mask_mm3"], axis=0)

# 5) salva CSV completo
out_path = f"csvs/{ab}/feat_rad_all.csv"
merged.to_csv(out_path, index=False)

print(f"Salvo: {out_path}")
print(f"shape: {merged.shape}")
print(f"colunas volumetricas: {VOL_FEAT_COLS}")
print(f"normalizadas pelo ICV: {cols_to_norm}")

out_path


Salvo: /mnt/study-data/pgirardi/graphs/csvs/longitudinal_4_groups/feat_rad_all.csv
shape: (31500, 121)
colunas volumetricas: ['mask_mm3', 'gm_mm3', 'gm_norm', 'wm_mm3', 'wm_norm', 'csf_mm3', 'csf_norm', 'tissues_mm3', 'tissues_norm']
normalizadas pelo ICV: ['original_firstorder_Energy', 'original_firstorder_TotalEnergy', 'original_shape_MeshVolume', 'original_shape_VoxelVolume', 'original_shape_SurfaceArea', 'original_shape_LeastAxisLength', 'original_shape_MajorAxisLength', 'original_shape_MinorAxisLength', 'original_shape_Maximum2DDiameterColumn', 'original_shape_Maximum2DDiameterRow', 'original_shape_Maximum2DDiameterSlice', 'original_shape_Maximum3DDiameter', 'mask_mm3', 'gm_mm3', 'wm_mm3', 'csf_mm3', 'tissues_mm3']


'/mnt/study-data/pgirardi/graphs/csvs/longitudinal_4_groups/feat_rad_all.csv'

In [36]:
import pandas as pd

feat_rad_all = pd.read_csv("csvs/longitudinal_4_groups/feat_rad_all.csv")

print(feat_rad_all.columns.tolist())

['ID_IMG', 'roi', 'side', 'label', 'original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Median', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_Autocorrelation', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_glcm_Correlation', 'original_glcm_DifferenceAverage', 'original_glcm_DifferenceEntropy', 'original_glcm_DifferenceVariance', 'original_glcm_Id', 'original_glcm_Idm', 'origina

In [37]:
import pandas as pd

ab = "longitudinal_4_groups"
BASE = f"csvs/{ab}"
LONGITUDINAL = "csvs/adnimerged_longitudinal.csv"

radiomics_merged_path = f"{BASE}/feat_rad_all.csv"
radiomics_merge = pd.read_csv(radiomics_merged_path)
radiomics_merge["ID_IMG"] = radiomics_merge["ID_IMG"].astype(str).str.strip()

COHORT_COLS = ["ID_IMG", "ID_PT", "GROUP", "SEX", "AGE", "MRI_DATE", "DIAG", "slot"]
TECH_COLS = [
    "ID_IMG",
    "FIELD_STRENGTH",
    "MANUFACTURER",
    "MFG_MODEL",
    "MMSE_SCORE",
    "CDR_GLOBAL",
    "ADAS_SCORE",
    "FAQ_SCORE",
]

longitudinal = pd.read_csv(LONGITUDINAL)
meta_cols = list(dict.fromkeys(COHORT_COLS + TECH_COLS))
missing = [c for c in meta_cols if c not in longitudinal.columns]
if missing:
    raise KeyError(f"Colunas ausentes em {LONGITUDINAL}: {missing}")

meta_sub = (
    longitudinal[meta_cols]
    .assign(ID_IMG=lambda d: d["ID_IMG"].astype(str).str.strip())
    .drop_duplicates(subset=["ID_IMG"], keep="last")
)

merge = radiomics_merge.merge(meta_sub, on="ID_IMG", how="left", validate="many_to_one")

batch_cols = ["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH"]
merge["batch"] = merge[batch_cols].astype(str).agg("_".join, axis=1)

meta_check = ["ID_PT", "GROUP", "SEX", "AGE", "MRI_DATE", "DIAG"]
print(f"Shape merge: {merge.shape}")
print(f"Sem meta cohort: {int(merge[meta_check].isna().any(axis=1).sum())}")
print(f"Sem scanner:     {int(merge[batch_cols].isna().any(axis=1).sum())}")
print(f"\nGROUP:\n{merge['GROUP'].value_counts(dropna=False)}")

META_ORDER = [
    "ID_IMG", "roi", "side", "label",
    "ID_PT", "GROUP", "SEX", "AGE", "MRI_DATE", "DIAG", "slot",
    "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE",
    "batch",
]
VOL_FEAT_COLS = [
    "mask_mm3", "gm_mm3", "gm_norm", "wm_mm3", "wm_norm",
    "csf_mm3", "csf_norm", "tissues_mm3", "tissues_norm",
]

prefix = [c for c in META_ORDER if c in merge.columns]
radiomic_cols = [c for c in merge.columns if c.startswith("original_")]
vol_cols = [c for c in VOL_FEAT_COLS if c in merge.columns]
icv_cols = [c for c in merge.columns if c == "ICV_mask_mm3"]
known = set(prefix) | set(radiomic_cols) | set(vol_cols) | set(icv_cols)
extra = [c for c in merge.columns if c not in known]

merge = merge[prefix + radiomic_cols + vol_cols + icv_cols + extra]

out_path = f"{BASE}/feat_rad_all.csv"
merge.to_csv(out_path, index=False)
print(f"Salvo: {out_path}")
print(f"  radiomics={len(radiomic_cols)} | volumetric={len(vol_cols)} | ICV={len(icv_cols)}")


Shape merge: (31500, 135)
Sem meta cohort: 0
Sem scanner:     0

GROUP:
GROUP
sMCI    23820
pMCI     7680
Name: count, dtype: int64
Salvo: /mnt/study-data/pgirardi/graphs/csvs/longitudinal_4_groups/feat_rad_all.csv
  radiomics=107 | volumetric=9 | ICV=1


In [38]:
import pandas as pd

feat_rad_all = pd.read_csv("csvs/longitudinal_4_groups/feat_rad_all.csv")

print(feat_rad_all.columns.tolist())

['ID_IMG', 'roi', 'side', 'label', 'ID_PT', 'GROUP', 'SEX', 'AGE', 'MRI_DATE', 'DIAG', 'FIELD_STRENGTH', 'MANUFACTURER', 'MFG_MODEL', 'MMSE_SCORE', 'CDR_GLOBAL', 'ADAS_SCORE', 'FAQ_SCORE', 'batch', 'original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Median', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_Autocorrelation', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_glcm_Correl

In [39]:
import pandas as pd

ab = "longitudinal_4_groups"
BASE = f"csvs/{ab}"
LONGITUDINAL = "csvs/adnimerged_longitudinal.csv"

PATH_DISP = f"{BASE}/features_displacement.csv"
PATH_OUT = f"{BASE}/feat_disp_all.csv"

MERGE_KEYS = ["ID_IMG", "roi", "side", "label"]

META_COLS = {
    "ID_PT", "ID_IMG", "DIAG", "GROUP", "SEX", "AGE", "MRI_DATE", "slot",
    "ref_tag", "roi", "side", "label",
    "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE", "batch",
}

TECH_COLS = [
    "ID_IMG", "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE",
]


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ID_IMG"] = out["ID_IMG"].astype(str).str.strip()
    out["roi"] = out["roi"].astype(str).str.strip()
    out["side"] = out["side"].astype(str).str.strip()
    out["label"] = out["label"].astype(str).str.strip()
    return out


df_disp = normalize_keys(pd.read_csv(PATH_DISP))

META_EXTRA_COLS = ["ID_IMG", "slot"] + [
    c for c in TECH_COLS if c != "ID_IMG"
]

meta_sub = (
    pd.read_csv(LONGITUDINAL)[META_EXTRA_COLS]
    .assign(ID_IMG=lambda d: d["ID_IMG"].astype(str).str.strip())
    .drop_duplicates(subset=["ID_IMG"], keep="last")
)

overlap = [c for c in meta_sub.columns if c in df_disp.columns and c != "ID_IMG"]
df_disp = df_disp.drop(columns=overlap, errors="ignore")

df_all = df_disp.merge(meta_sub, on="ID_IMG", how="left", validate="many_to_one")

batch_cols = ["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH"]
df_all["batch"] = df_all[batch_cols].astype(str).agg("_".join, axis=1)

META_ORDER = [
    "ID_IMG", "roi", "side", "label",
    "ID_PT", "GROUP", "SEX", "AGE", "MRI_DATE", "DIAG", "slot", "ref_tag",
    "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE",
    "batch",
]
prefix = [c for c in META_ORDER if c in df_all.columns]
feature_cols = [c for c in df_all.columns if c not in prefix]
df_all = df_all[prefix + feature_cols]

df_all.to_csv(PATH_OUT, index=False)
print("entrada:", df_disp.shape)
print("Shape final:", df_all.shape)
print("Sem scanner:", int(df_all[batch_cols].isna().any(axis=1).sum()))
print(f"Salvo: {PATH_OUT}")

legacy: (31500, 122)
article: (31500, 26)
Shape final: (31500, 139)
Sem scanner: 0


In [40]:
import pandas as pd

ab = "longitudinal_4_groups"
BASE = f"csvs/{ab}"

disp_path = f"{BASE}/feat_disp_all.csv"
rad_path = f"{BASE}/feat_rad_all.csv"
out_all_unit = f"{BASE}/feat_merge_all.csv"

MERGE_KEYS = ["ID_IMG", "roi", "side", "label"]
# Metadados clínicos vêm do displacement; rad fica com radiomico + ICV + técnicas + batch
# OVERLAP_META = {"ID_PT", "SEX", "DIAG", "GROUP", "AGE", "MRI_DATE"}
OVERLAP_META = {
    "ID_PT", "SEX", "DIAG", "GROUP", "AGE", "MRI_DATE", "slot",
    "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE", "batch",
}

def normalize_merge_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ID_IMG"] = out["ID_IMG"].astype(str).str.strip()
    out["roi"] = out["roi"].astype(str).str.strip()
    out["side"] = out["side"].astype(str).str.strip()
    out["label"] = out["label"].astype(str).str.strip()
    return out

rad = pd.read_csv(rad_path)
rad = rad.drop(columns=[c for c in OVERLAP_META if c in rad.columns], errors="ignore")
rad = normalize_merge_keys(rad)

disp = pd.read_csv(disp_path)
disp = normalize_merge_keys(disp)

all_unit = rad.merge(
    disp,
    on=MERGE_KEYS,
    how="left",
    validate="one_to_one",
)

if "MRI_DATE" in all_unit.columns:
    all_unit["MRI_DATE"] = (
        pd.to_datetime(all_unit["MRI_DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
    )


def reorder_all_features_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Demográficas/técnicas primeiro; depois radiomico, ICV e deslocamento."""
    # meta_cols = [
    #     "ID_IMG",
    #     "ID_PT",
    #     "GROUP",
    #     "SEX",
    #     "AGE",
    #     "MRI_DATE",
    #     "roi",
    #     "side",
    #     "label",
    #     "FIELD_STRENGTH",
    #     "MANUFACTURER",
    #     "MFG_MODEL",
    #     "batch",
    #     "DIAG",
    #     "MMSE_SCORE",
    #     "CDR_GLOBAL",
    #     "ADAS_SCORE",
    #     "FAQ_SCORE",
    # ]
    meta_cols = [
    "ID_IMG", "roi", "side", "label",
    "ID_PT", "GROUP", "SEX", "AGE", "MRI_DATE", "DIAG", "slot", "ref_tag",
    "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL",
    "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE", "batch",
]

    # features_displacement.csv = legado (percentis) + momentos do artigo
    disp_prefixes = (
        "centroid_",
        "logjac_",
        "mag_",
        "div_",
        "ux_",
        "uy_",
        "uz_",
        "curlmag_",
        "strain_trace_",
        "strain_det_",
        "strain_fro_",
        "strain_vol_",
        "strain_dev_norm_",
        "strain_shear_max_",
        "strain_l1_",
        "strain_l2_",
        "strain_l3_",
        "strain_shear_ratio_",
        "strain_shear_energy_",
    )
    stat_suffixes = (
        "_n",
        "_mean",
        "_std",
        "_p05",
        "_p50",
        "_p95",
        "_variance",
        "_skewness",
        "_kurtosis",
    )

    VOL_FEAT_COLS = [
        "mask_mm3", "gm_mm3", "gm_norm", "wm_mm3", "wm_norm",
        "csf_mm3", "csf_norm", "tissues_mm3", "tissues_norm",
    ]

    cols = list(df.columns)
    prefix = [c for c in meta_cols if c in cols]
    radiomic = [c for c in cols if c.startswith("original_")]
    volumetric = [c for c in VOL_FEAT_COLS if c in cols]
    icv = [c for c in cols if c == "ICV_mask_mm3"]
    used = set(prefix) | set(radiomic) | set(volumetric) | set(icv)
    disp_cols = [c for c in cols if c not in used]

    def _disp_key(name: str) -> tuple:
        for i, p in enumerate(disp_prefixes):
            if name.startswith(p):
                base = p.rstrip("_")
                for j, suf in enumerate(stat_suffixes):
                    if name.endswith(suf) and name == base + suf:
                        return (i, j, name)
                return (i, 99, name)
        return (len(disp_prefixes), 0, name)

    disp_cols = sorted(disp_cols, key=_disp_key)
    ordered = prefix + radiomic + volumetric + icv + disp_cols
    extra = [c for c in cols if c not in ordered]
    if extra:
        print(f"Aviso: colunas extras ao final: {extra}")
        ordered = ordered + extra
    return df.loc[:, ordered]


all_unit = reorder_all_features_columns(all_unit)

all_unit.to_csv(out_all_unit, index=False)
print("Salvo:", out_all_unit)

print("feat rad:", rad.shape)
print("feat disp:", disp.shape)
print("feat all:", all_unit.shape)

Salvo: /mnt/study-data/pgirardi/graphs/csvs/longitudinal_4_groups/feat_merge_all.csv
feat rad: (31500, 121)
feat disp: (31500, 139)
feat all: (31500, 256)


In [41]:
import pandas as pd

feat_merge_all = pd.read_csv("csvs/longitudinal_4_groups/feat_merge_all.csv")

print(feat_merge_all.columns.tolist())


['ID_IMG', 'roi', 'side', 'label', 'ID_PT', 'GROUP', 'SEX', 'AGE', 'MRI_DATE', 'DIAG', 'ref_tag', 'FIELD_STRENGTH', 'MANUFACTURER', 'MFG_MODEL', 'MMSE_SCORE', 'CDR_GLOBAL', 'ADAS_SCORE', 'FAQ_SCORE', 'batch', 'original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Median', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_Autocorrelation', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_

# Filtra as rois

In [43]:
import pandas as pd
from ablation_prep import ROI_FILTER_DEFAULT, filter_rois

ROI_FILTER = ROI_FILTER_DEFAULT  # parametrizável: "hippocampus", lista, etc.

ab = "longitudinal_4_groups"
BASE = f"csvs/{ab}"

feat_merge_all_df = pd.read_csv(f"{BASE}/feat_merge_all.csv")
feat_rad_all_df = pd.read_csv(f"{BASE}/feat_rad_all.csv")
feat_disp_all_df = pd.read_csv(f"{BASE}/feat_disp_all.csv")

feat_merge_all_hipp_df = filter_rois(feat_merge_all_df, ROI_FILTER)
feat_rad_all_hipp_df = filter_rois(feat_rad_all_df, ROI_FILTER)
feat_disp_all_hipp_df = filter_rois(feat_disp_all_df, ROI_FILTER)

# Long (ComBat / ablation) — não sobrescrever com wide
feat_merge_all_hipp_df.to_csv(f"{BASE}/feat_merge_all_{ROI_FILTER}_long.csv", index=False)
feat_rad_all_hipp_df.to_csv(f"{BASE}/feat_rad_all_{ROI_FILTER}_long.csv", index=False)
feat_disp_all_hipp_df.to_csv(f"{BASE}/feat_disp_all_{ROI_FILTER}_long.csv", index=False)

# Aliases legados (delta / filtros downstream)
feat_merge_all_hipp_df.to_csv(f"{BASE}/feat_merge_all_hipp.csv", index=False)
feat_rad_all_hipp_df.to_csv(f"{BASE}/feat_rad_all_hipp.csv", index=False)
feat_disp_all_hipp_df.to_csv(f"{BASE}/feat_disp_all_hipp.csv", index=False)

print(f"ROI={ROI_FILTER} | long salvos em feat_*_all_{ROI_FILTER}_long.csv")



# Cada linha um conjunto com 3 imagens

In [46]:
# 1 linha/paciente; colunas wide {roi}_{L|R}_{T1|T2|T3}_{feat}

from pathlib import Path

import pandas as pd
from ablation_prep import ROI_FILTER_DEFAULT, pivot_long_to_wide

ROI_FILTER = ROI_FILTER_DEFAULT
ab = "longitudinal_4_groups"
BASE = Path(f"csvs/{ab}")

WIDE_TARGETS = {
    "rad": (
        BASE / f"feat_rad_all_{ROI_FILTER}_long.csv",
        BASE / f"feat_rad_all_{ROI_FILTER}_wide.csv",
    ),
    "disp": (
        BASE / f"feat_disp_all_{ROI_FILTER}_long.csv",
        BASE / f"feat_disp_all_{ROI_FILTER}_wide.csv",
    ),
    "merge": (
        BASE / f"feat_merge_all_{ROI_FILTER}_long.csv",
        BASE / f"feat_merge_all_{ROI_FILTER}_wide.csv",
    ),
}

for label, (path_in, path_out) in WIDE_TARGETS.items():
    df_in = pd.read_csv(path_in)
    wide = pivot_long_to_wide(df_in)
    wide.to_csv(path_out, index=False)
    n_feat = wide.shape[1] - 3
    print(f"[{label}] {path_out}")
    print(f"  pacientes={len(wide)} | colunas={wide.shape[1]} | features base≈{n_feat // 6}")


Teste 2 salvo: csvs/longitudinal_4_groups/feat_disp_all_hipp.csv
  pacientes=525 | colunas=723 | features base=120


In [ ]:
# Export vol / texture / disp / all para ablation (long + wide filtrados)

from pathlib import Path

from ablation_prep import ROI_FILTER_DEFAULT, export_ablation_datasets

ROI_FILTER = ROI_FILTER_DEFAULT
BASE = Path("csvs/longitudinal_4_groups")

ablation_paths = export_ablation_datasets(BASE, roi=ROI_FILTER)
ablation_paths

# Filtragem ICV+Textura

In [85]:
import re
from pathlib import Path

import pandas as pd

# ── Configuração ──────────────────────────────────────────────────────────────

META_COLS = ["ID_PT", "GROUP", "SEX"]
BASE_DIR = Path("csvs/longitudinal_4_groups")

# Extrai o nome da feature após hippocampus_{L|R}_{T1|T2|T3}_
WIDE_FEAT_RE = re.compile(r"^hippocampus_[LR]_T[123]_(.+)$")

# ── Radiomics: o que remover ──────────────────────────────────────────────────

ICV_RE = re.compile(r"ICV_mask_mm3$")
TEXTURE_RE = re.compile(r"original_(glcm|gldm|glrlm|glszm|ngtdm)_")

# first-order redundantes / instáveis (sufixo após original_firstorder_)
RAD_FIRSTORDER_DROP = {
    "Median",
    "RootMeanSquared",
    "Minimum",
    "Maximum",
    "Range",
    "MeanAbsoluteDeviation",
    "RobustMeanAbsoluteDeviation",
    "TotalEnergy",
    # opcionais (recomendados na discussão anterior):
    # "Kurtosis",
    # "Skewness",
    # "Entropy",
    # "Uniformity",
    # "Energy",  # manter Variance
}

# shape redundante (sufixo após original_shape_)
RAD_SHAPE_DROP = {
    "Maximum2DDiameterColumn",
    "Maximum2DDiameterRow",
    "Maximum2DDiameterSlice",
    "Maximum3DDiameter",
    # opcionais:
    "LeastAxisLength",
    "MajorAxisLength",
    "MinorAxisLength",
    "VoxelVolume",  # manter MeshVolume
}

# ── Displacement: o que remover ───────────────────────────────────────────────

# Famílias inteiras (prefixo da feature base, ex.: logjac_mean → logjac_)
DISP_PREFIX_DROP = (
    "centroid_",
    "ux_",
    "uy_",
    "uz_",
    "div_",
    "curlmag_",
    "mag_",
    "strain_trace_",
    "strain_vol_",
    "strain_det_",
    "strain_shear_ratio_",
    "strain_shear_energy_",
    "strain_shear_max_",
    "strain_l1_",
    "strain_l2_",
    "strain_l3_",

)

# Estatísticas instáveis / QC (sufixo)
DISP_STAT_DROP = (
    "_n",
    "_variance",
    # "_skewness",
    # "_kurtosis",
    
)

# Estatísticas redundantes se você mantiver mean+std
# (descomente se quiser só mean/std)
DISP_STAT_DROP = DISP_STAT_DROP + ("_p05", "_p50", "_p95")


# ── Helpers ───────────────────────────────────────────────────────────────────

def _is_meta(col: str) -> bool:
    return col in META_COLS


def _feature_name(col: str) -> str | None:
    """Retorna a parte da feature sem o prefixo temporal wide."""
    m = WIDE_FEAT_RE.match(col)
    return m.group(1) if m else None


def _should_drop_radiomics_col(
    col: str,
    *,
    drop_texture: bool,
    drop_redundant: bool,
) -> bool:
    if _is_meta(col):
        return False

    feat = _feature_name(col)
    if feat is None:
        return False  # não mexe em colunas fora do padrão wide

    if ICV_RE.search(feat):
        return True

    if drop_texture and TEXTURE_RE.search(feat):
        return True

    if not drop_redundant:
        return False

    if feat.startswith("original_firstorder_"):
        suffix = feat.removeprefix("original_firstorder_")
        return suffix in RAD_FIRSTORDER_DROP

    if feat.startswith("original_shape_"):
        suffix = feat.removeprefix("original_shape_")
        return suffix in RAD_SHAPE_DROP

    return False


def _should_drop_displacement_col(col: str) -> bool:
    if _is_meta(col):
        return False

    feat = _feature_name(col)
    if feat is None:
        return False

    # remove família inteira (centroid_x, logjac_mean, strain_trace_std, ...)
    if any(feat.startswith(prefix) for prefix in DISP_PREFIX_DROP):
        return True

    # remove estatísticas indesejadas em qualquer família restante
    if any(feat.endswith(suffix) for suffix in DISP_STAT_DROP):
        return True

    return False


def _drop_with_report(df: pd.DataFrame, predicate, label: str) -> tuple[pd.DataFrame, list[str]]:
    drop_cols = [c for c in df.columns if predicate(c)]
    print(f"[{label}] removidas {len(drop_cols)} colunas | restam {df.shape[1] - len(drop_cols)}")
    return df.drop(columns=drop_cols), drop_cols


# ── 1) Radiomics ──────────────────────────────────────────────────────────────

def filter_radiomics_df(
    df: pd.DataFrame,
    *,
    drop_texture: bool = True,
    drop_redundant: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filtra colunas radiomics no CSV wide.

    Remove:
      - ICV_mask_mm3
      - textura (glcm, gldm, glrlm, glszm, ngtdm) [opcional]
      - first-order / shape redundantes [opcional]
    """
    def pred(col):
        return _should_drop_radiomics_col(
            col,
            drop_texture=drop_texture,
            drop_redundant=drop_redundant,
        )

    if verbose:
        out, dropped = _drop_with_report(df, pred, "radiomics")
        return out
    return df.drop(columns=[c for c in df.columns if pred(c)])


# ── 2) Displacement ───────────────────────────────────────────────────────────

def filter_displacement_df(
    df: pd.DataFrame,
    *,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filtra colunas de deslocamento no CSV wide.

    Remove:
      - centroid, ux/uy/uz, div, curlmag, mag
      - strain_trace/vol/det, strain_shear_ratio/energy
      - stats _n, _variance, _skewness, _kurtosis
    """
    if verbose:
        out, _ = _drop_with_report(df, _should_drop_displacement_col, "displacement")
        return out
    return df.drop(columns=[c for c in df.columns if _should_drop_displacement_col(c)])


# ── 3) Orquestrador ───────────────────────────────────────────────────────────

def build_filtered_datasets(
    df_rad: pd.DataFrame,
    df_disp: pd.DataFrame,
    *,
    drop_texture: bool = True,
    drop_rad_redundant: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Aplica filtros em radiomics e displacement e monta o merge filtrado.
    """
    rad_f = filter_radiomics_df(
        df_rad,
        drop_texture=drop_texture,
        drop_redundant=drop_rad_redundant,
    )
    disp_f = filter_displacement_df(df_disp)

    rad_cols = [c for c in rad_f.columns if c not in META_COLS]
    disp_cols = [c for c in disp_f.columns if c not in META_COLS]

    merge_f = rad_f[META_COLS].join(rad_f[rad_cols]).join(disp_f[disp_cols])

    print(
        f"[merge] pacientes={len(merge_f)} | "
        f"colunas={merge_f.shape[1]} | "
        f"rad={len(rad_cols)} | disp={len(disp_cols)}"
    )

    return {
        "rad": rad_f,
        "disp": disp_f,
        "merge": merge_f,
    }


def save_filtered_csvs(
    datasets: dict[str, pd.DataFrame],
    *,
    suffix: str = "filtered",
    out_dir: Path = BASE_DIR,
) -> dict[str, Path]:
    """Salva rad, disp e merge com sufixo no nome do arquivo."""
    paths = {
        "rad": out_dir / f"feat_rad_all_hipp_{suffix}.csv",
        "disp": out_dir / f"feat_disp_all_hipp_{suffix}.csv",
        "merge": out_dir / f"feat_merge_all_hipp_{suffix}.csv",
    }
    for key, path in paths.items():
        datasets[key].to_csv(path, index=False)
        print(f"Salvo: {path}")
    return paths


# ── Execução ──────────────────────────────────────────────────────────────────

df_rad = pd.read_csv(BASE_DIR / "feat_rad_all_hipp.csv")
df_disp = pd.read_csv(BASE_DIR / "feat_disp_all_hipp.csv")

datasets = build_filtered_datasets(
    df_rad,
    df_disp,
    drop_texture=True,       # False → mantém textura, só remove ICV + redundantes
    drop_rad_redundant=True,
)

save_filtered_csvs(datasets, suffix="filtered")

[radiomics] removidas 552 colunas | restam 153
[displacement] removidas 660 colunas | restam 63
[merge] pacientes=525 | colunas=213 | rad=150 | disp=60
Salvo: csvs/longitudinal_4_groups/feat_rad_all_hipp_filtered.csv
Salvo: csvs/longitudinal_4_groups/feat_disp_all_hipp_filtered.csv
Salvo: csvs/longitudinal_4_groups/feat_merge_all_hipp_filtered.csv


{'rad': PosixPath('csvs/longitudinal_4_groups/feat_rad_all_hipp_filtered.csv'),
 'disp': PosixPath('csvs/longitudinal_4_groups/feat_disp_all_hipp_filtered.csv'),
 'merge': PosixPath('csvs/longitudinal_4_groups/feat_merge_all_hipp_filtered.csv')}

# Filtro apenas volume

In [1]:
import re

import pandas as pd

merge = pd.read_csv("csvs/longitudinal_4_groups/feat_merge_all_hipp.csv")

# estático: hippocampus_L_T1_gm_norm
WIDE_VOL_RE = re.compile(r"^hippocampus_[LR]_T[123]_(.+)$")

VOL_FEATURE_SUFFIXES = {
    # "original_shape_MeshVolume",
    "gm_norm",
    # "wm_norm",
    # "csf_norm", #"gm_mm3","wm_mm3", "csf_mm3",
    # opcionais: "mask_mm3", "tissues_mm3", "tissues_norm",
}


def select_volume_columns(columns) -> list[str]:
    """MeshVolume + volumes teciduais (gm/wm/csf, absolutos e norm)."""
    out: list[str] = []
    for col in columns:
        m = WIDE_VOL_RE.match(col)
        if m and m.group(1) in VOL_FEATURE_SUFFIXES:
            out.append(col)
    return out

META_COLS = ["ID_PT", "GROUP", "SEX"]
vol_cols = select_volume_columns(merge.columns)
df_vol = merge[META_COLS + vol_cols]

print(f"{len(vol_cols)} colunas de volume:")
print(vol_cols)

out_path = "csvs/longitudinal_4_groups/feat_gm_volume_only.csv"
df_vol.to_csv(out_path, index=False)

print(f"Salvo: {out_path} | shape: {df_vol.shape}")

6 colunas de volume:
['hippocampus_L_T1_gm_norm', 'hippocampus_L_T2_gm_norm', 'hippocampus_L_T3_gm_norm', 'hippocampus_R_T1_gm_norm', 'hippocampus_R_T2_gm_norm', 'hippocampus_R_T3_gm_norm']
Salvo: csvs/longitudinal_4_groups/feat_gm_volume_only.csv | shape: (525, 9)


# Geração dos deltas

In [78]:
# Deltas longitudinais a partir dos CSVs wide (T1, T2, T3 → d12, d23, d13)
#
# Modo relativo (padrão): taxa de mudança em relação ao tempo de referência
#   d12 = (f_T2 − f_T1) / f_T1
#   d23 = (f_T3 − f_T2) / f_T2
#   d13 = (f_T3 − f_T1) / f_T1
#
# Modo absoluto (DELTA_MODE="absolute"): diferença simples f_dest − f_orig
#
# Entrada: feat_*_all_hipp.csv (sem filtros). Filtros aplicados depois.

import re
from pathlib import Path

import numpy as np
import pandas as pd

META_COLS = ["ID_PT", "GROUP", "SEX"]
BASE_DIR = Path("csvs/longitudinal_4_groups")

# "relative" | "absolute"
DELTA_MODE = "absolute"
# Denominador |f_ref| < EPS → NaN (evita explosão numérica)
DELTA_EPS = 1e-8

WIDE_COL_RE = re.compile(r"^hippocampus_([LR])_(T[123])_(.+)$")

# (tempo_referência, tempo_destino, rótulo da coluna delta)
DELTA_WINDOWS = (
    ("T1", "T2", "d12"),
    ("T2", "T3", "d23"),
    # ("T1", "T3", "d13"),
)


def _delta_values(v_from: pd.Series, v_to: pd.Series, mode: str) -> pd.Series:
    """Calcula delta absoluto ou relativo entre dois tempos."""
    v_from = pd.to_numeric(v_from, errors="coerce")
    v_to = pd.to_numeric(v_to, errors="coerce")
    diff = v_to - v_from
    if mode == "absolute":
        return diff
    if mode == "relative":
        denom = v_from.copy()
        denom = denom.where(denom.abs() >= DELTA_EPS)
        return diff / denom
    raise ValueError(f"DELTA_MODE inválido: {mode!r} (use 'relative' ou 'absolute')")


def compute_deltas_from_wide(
    df: pd.DataFrame,
    *,
    mode: str = DELTA_MODE,
) -> pd.DataFrame:
    """
    Converte colunas hippocampus_{L|R}_{T1|T2|T3}_{feat}
    em hippocampus_{L|R}_{d12|d23|d13}_{feat}.

    No modo relativo, cada delta é a variação percentual (futuro − passado) / passado.
    """
    missing_meta = [c for c in META_COLS if c not in df.columns]
    if missing_meta:
        raise KeyError(f"Colunas meta ausentes: {missing_meta}")

    # (side, feat) -> {T1: nome_coluna, T2: ..., T3: ...}
    time_index: dict[tuple[str, str], dict[str, str]] = {}
    skipped_cols: list[str] = []

    for col in df.columns:
        if col in META_COLS:
            continue
        m = WIDE_COL_RE.match(col)
        if not m:
            skipped_cols.append(col)
            continue
        side, time, feat = m.group(1), m.group(2), m.group(3)
        time_index.setdefault((side, feat), {})[time] = col

    delta_series: dict[str, pd.Series] = {}
    incomplete = 0

    for (side, feat), times in sorted(time_index.items()):
        for t_from, t_to, label in DELTA_WINDOWS:
            if t_from not in times or t_to not in times:
                incomplete += 1
                continue
            out_col = f"hippocampus_{side}_{label}_{feat}"
            delta_series[out_col] = _delta_values(
                df[times[t_from]],
                df[times[t_to]],
                mode=mode,
            )

    out = df[META_COLS].copy()
    if delta_series:
        out = pd.concat([out, pd.DataFrame(delta_series)], axis=1)

    n_nan = int(out.iloc[:, len(META_COLS):].isna().sum().sum()) if delta_series else 0
    print(
        f"  modo={mode} | blocos (side×feat): {len(time_index)} | "
        f"deltas gerados: {len(delta_series)} | "
        f"janelas incompletas: {incomplete} | NaN: {n_nan}"
    )
    if skipped_cols:
        print(f"  aviso: {len(skipped_cols)} colunas fora do padrão wide ignoradas")

    return out


def build_delta_csv(
    path_in: Path,
    path_out: Path,
    label: str,
    *,
    mode: str = DELTA_MODE,
) -> pd.DataFrame:
    print(f"\n[{label}] {path_in.name}")
    df = pd.read_csv(path_in)
    delta_df = compute_deltas_from_wide(df, mode=mode)
    delta_df.to_csv(path_out, index=False)
    n_feat = delta_df.shape[1] - len(META_COLS)
    print(
        f"  → {path_out.name} | pacientes={len(delta_df)} | "
        f"features={n_feat} | colunas={delta_df.shape[1]}"
    )
    return delta_df


DATASETS = {
    "rad": (
        BASE_DIR / "feat_rad_all_hipp.csv",
        BASE_DIR / "feat_rad_all_hipp_delta.csv",
    ),
    "disp": (
        BASE_DIR / "feat_disp_all_hipp.csv",
        BASE_DIR / "feat_disp_all_hipp_delta.csv",
    ),
    "merge": (
        BASE_DIR / "feat_merge_all_hipp.csv",
        BASE_DIR / "feat_merge_all_hipp_delta.csv",
    ),
}

delta_datasets: dict[str, pd.DataFrame] = {}
for key, (path_in, path_out) in DATASETS.items():
    if not path_in.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path_in}")
    delta_datasets[key] = build_delta_csv(path_in, path_out, key)

print("\n=== Resumo ===")
for key, df in delta_datasets.items():
    print(f"  {key:5s}: {df.shape[0]} pacientes | {df.shape[1] - len(META_COLS)} features delta")


[rad] feat_rad_all_hipp.csv
  modo=absolute | blocos (side×feat): 234 | deltas gerados: 468 | janelas incompletas: 0 | NaN: 0
  → feat_rad_all_hipp_delta.csv | pacientes=525 | features=468 | colunas=471

[disp] feat_disp_all_hipp.csv
  modo=absolute | blocos (side×feat): 240 | deltas gerados: 480 | janelas incompletas: 0 | NaN: 0
  → feat_disp_all_hipp_delta.csv | pacientes=525 | features=480 | colunas=483

[merge] feat_merge_all_hipp.csv
  modo=absolute | blocos (side×feat): 474 | deltas gerados: 948 | janelas incompletas: 0 | NaN: 0
  → feat_merge_all_hipp_delta.csv | pacientes=525 | features=948 | colunas=951

=== Resumo ===
  rad  : 525 pacientes | 468 features delta
  disp : 525 pacientes | 480 features delta
  merge: 525 pacientes | 948 features delta


In [79]:
import re

import pandas as pd

META_COLS = ["ID_PT", "GROUP", "SEX"]

# static: hippocampus_L_T1_gm_norm  |  delta: hippocampus_L_d12_gm_norm
WIDE_VOL_RE = re.compile(
    r"^hippocampus_[LR]_(?:T[123]|d12|d23|d13)_(.+)$"
)

VOL_FEATURE_SUFFIXES = {
    "original_shape_MeshVolume",
    "gm_norm",
    "wm_norm",
    "csf_norm",
}


def select_volume_columns(columns) -> list[str]:
    """MeshVolume + frações teciduais (gm/wm/csf norm)."""
    out: list[str] = []
    for col in columns:
        m = WIDE_VOL_RE.match(col)
        if m and m.group(1) in VOL_FEATURE_SUFFIXES:
            out.append(col)
    return out


merge = pd.read_csv("csvs/longitudinal_4_groups/feat_merge_all_hipp_delta.csv")
vol_cols = select_volume_columns(merge.columns)
df_vol = merge[META_COLS + vol_cols]

print(f"{len(vol_cols)} colunas de volume:")
print(vol_cols)

out_path = "csvs/longitudinal_4_groups/feat_volume_only_delta.csv"
df_vol.to_csv(out_path, index=False)

print(f"Salvo: {out_path} | shape: {df_vol.shape}")

16 colunas de volume:
['hippocampus_L_d12_csf_norm', 'hippocampus_L_d23_csf_norm', 'hippocampus_L_d12_gm_norm', 'hippocampus_L_d23_gm_norm', 'hippocampus_L_d12_original_shape_MeshVolume', 'hippocampus_L_d23_original_shape_MeshVolume', 'hippocampus_L_d12_wm_norm', 'hippocampus_L_d23_wm_norm', 'hippocampus_R_d12_csf_norm', 'hippocampus_R_d23_csf_norm', 'hippocampus_R_d12_gm_norm', 'hippocampus_R_d23_gm_norm', 'hippocampus_R_d12_original_shape_MeshVolume', 'hippocampus_R_d23_original_shape_MeshVolume', 'hippocampus_R_d12_wm_norm', 'hippocampus_R_d23_wm_norm']
Salvo: csvs/longitudinal_4_groups/feat_volume_only_delta.csv | shape: (525, 19)


In [80]:
print(merge.columns.tolist())
print(df_vol.columns.tolist())

['ID_PT', 'GROUP', 'SEX', 'hippocampus_L_d12_ICV_mask_mm3', 'hippocampus_L_d23_ICV_mask_mm3', 'hippocampus_L_d12_centroid_x', 'hippocampus_L_d23_centroid_x', 'hippocampus_L_d12_centroid_y', 'hippocampus_L_d23_centroid_y', 'hippocampus_L_d12_centroid_z', 'hippocampus_L_d23_centroid_z', 'hippocampus_L_d12_csf_mm3', 'hippocampus_L_d23_csf_mm3', 'hippocampus_L_d12_csf_norm', 'hippocampus_L_d23_csf_norm', 'hippocampus_L_d12_curlmag_mean', 'hippocampus_L_d23_curlmag_mean', 'hippocampus_L_d12_curlmag_n', 'hippocampus_L_d23_curlmag_n', 'hippocampus_L_d12_curlmag_p05', 'hippocampus_L_d23_curlmag_p05', 'hippocampus_L_d12_curlmag_p50', 'hippocampus_L_d23_curlmag_p50', 'hippocampus_L_d12_curlmag_p95', 'hippocampus_L_d23_curlmag_p95', 'hippocampus_L_d12_curlmag_std', 'hippocampus_L_d23_curlmag_std', 'hippocampus_L_d12_div_mean', 'hippocampus_L_d23_div_mean', 'hippocampus_L_d12_div_n', 'hippocampus_L_d23_div_n', 'hippocampus_L_d12_div_p05', 'hippocampus_L_d23_div_p05', 'hippocampus_L_d12_div_p50', '

# Tarefas binárias (CN×AD, CN×sMCI, CN×pMCI, sMCI×pMCI)

Filtra os CSVs wide por par de grupos. `GROUP` permanece como rótulo string; encode binário na etapa de modelagem.

In [ ]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path("csvs/longitudinal_4_groups")

BINARY_TASKS = {
    "CN_vs_AD": ("CN", "AD"),
    "CN_vs_sMCI": ("CN", "sMCI"),
    "CN_vs_pMCI": ("CN", "pMCI"),
    "sMCI_vs_pMCI": ("sMCI", "pMCI"),
}

SOURCES = {
    "merge_static": BASE_DIR / "feat_merge_all_hipp.csv",
    "merge_delta": BASE_DIR / "feat_merge_all_hipp_delta.csv",
    "rad_static": BASE_DIR / "feat_rad_all_hipp.csv",
    "disp_static": BASE_DIR / "feat_disp_all_hipp.csv",
}

out_dir = BASE_DIR / "binary_tasks"
out_dir.mkdir(parents=True, exist_ok=True)

for src_name, src_path in SOURCES.items():
    if not src_path.is_file():
        print(f"[SKIP] {src_name}: {src_path} não encontrado")
        continue
    df = pd.read_csv(src_path)
    for task_name, (g0, g1) in BINARY_TASKS.items():
        sub = df[df["GROUP"].isin([g0, g1])].copy()
        out_path = out_dir / f"{task_name}_{src_name}.csv"
        sub.to_csv(out_path, index=False)
        print(
            f"{out_path.name}: n={len(sub)} | "
            f"{g0}={int((sub['GROUP'] == g0).sum())} | {g1}={int((sub['GROUP'] == g1).sum())}"
        )